# AutoARIMA Test Pipeline
This notebook adapts the LSTM pipeline for AutoARIMA, using the same recursive inference and validation logic.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os

In [2]:
# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/subset_set.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]

print(len(y))


# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 455
val_size = 153
forecast_horizon = 153

lookback_window = 30


df.head()

Loading data from ../dataset/subset_set.feather...
7610


,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,promo_value_DISC,promo_type_CIRC,promo_value_CIRC,promo_type_CIRE,promo_value_CIRE,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id
0,2021-01-23,26008,104,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
1,2021-01-23,921558,15,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
2,2021-01-23,213626,85,milknplant based bevs,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
3,2021-01-23,213625,44,milknplant based bevs,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
4,2021-01-23,213624,40,milknplant based bevs,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269


In [3]:
# DATA SPLITTING
train_size = 455
val_size = 153
forecast_horizon = 153
lookback_window = 30
print(df[TARGET_COL].describe())

count    7610.000000
mean       50.661761
std        30.226802
min         0.000000
25%        29.000000
50%        48.000000
75%        66.000000
max       504.000000
Name: value, dtype: float64


# AutoARIMA Experiment Function
This function matches the LSTM pipeline: same splits, recursive inference, and validation.

In [4]:
def autoarima_experiment_grid(df, target, item_id, store_id, train_size=432, val_size=153, forecast_window=153, save_plot_path=None):
    
    total_train_val = train_size + val_size
    train_slice = slice(-(total_train_val+forecast_window), -forecast_window)
    test_slice = slice(-forecast_window, None)
    train = df[target][train_slice].values
    test = df[target][test_slice].values
    model = auto_arima(train, 
                       start_p=0, d=None, start_q=2, 
                       max_p=5, max_d=2, max_q=5, start_P=1, D=None, start_Q=1, 
                       max_P=2, max_D=1, max_Q=2,seasonal=False, 
                       stepwise=True, suppress_warnings=True)
    # Recursive forecast (start from end of val, forecast one step at a time)
    forecast = model.predict(n_periods=forecast_window)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    # Save all plots in grid_search_plots directory
    plot_dir = f'grid_search_plots'
    os.makedirs(plot_dir, exist_ok=True)
    plot_filename = f'{plot_dir}/autoarima_item{item_id}_store{store_id}.png'
    if save_plot_path:
        plt.figure(figsize=(12,6))
        plt.plot(range(len(train)), train, label='Train')
        plt.plot(range(len(train), len(train)+len(test)), test, label='Test')
        plt.plot(range(len(train), len(train)+len(test)), forecast, label='Forecast')
        plt.title(f'AutoARIMA Forecast (Item={item_id}, Store={store_id})')
        plt.legend()
        plt.savefig(plot_filename)
        plt.close()
    return rmse, mae, model.order, plot_filename


# Grid Search Loop for Products and Seeds
This cell runs AutoARIMA for each product and seed, saving results and plots.

In [5]:
target_products = [921558, 26008]
if target_products:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    products = df[['item_id', 'store_id']].drop_duplicates().values
results = []
os.makedirs('grid_search_plots', exist_ok=True)

for item_id, store_id in products:
    df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
    df_product = df_product.reset_index(drop=True)
    df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
    df_product = df_product.sort_values(DATE_COL)
    df_product = df_product.reset_index(drop=True)
    plot_filename = f'grid_search_plots/autoarima_item{item_id}_store{store_id}.png'
    rmse, mae, order, plot_path = autoarima_experiment_grid(
        df=df_product,
        target=TARGET_COL,
        item_id=item_id,
        store_id=store_id,
        train_size=train_size,
        val_size=val_size,
        forecast_window=forecast_horizon,
        save_plot_path=plot_filename
    )
    results.append({
        'item_id': item_id,
        'store_id': store_id,
        'order': order,
        'rmse': rmse,
        'mae': mae,
        'plot_path': plot_filename
    })
results_df = pd.DataFrame(results)
results_df.to_csv('autoarima_grid_search_results.csv', index=False)